In [1]:
# an example of how ON/OFF neruons were classified 

import os
import sys
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path

REPO_ROOT = next(
    path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (path / "pyproject.toml").is_file()
)
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))
from Functions import *

import warnings
warnings.filterwarnings('ignore')

/opt/anaconda3/envs/LLM/lib/python3.12/site-packages/oasis/functions.py:13: UserWarning: Could not find cvxpy. Don't worry, you can still use OASIS, just not the slower interior point methods we compared to in the papers.
  warn("Could not find cvxpy. Don't worry, you can still use OASIS, " +


In [2]:
# prep info
root_path = os.path.join(REPO_ROOT, 'data', 'OFT_Data')

intervalLength = 20 # chunk must contain more than ~ frames
gapMax = 10 # if gap between two adjacent Chunks less than gapMax, merge the two
if_CombineAdjacentChunks = True

checkChance = True # if check the neuron acturally fire in Chunks
Fpercentage = 0.6 # % fire Chunks
SBpercentage = 80 # in one chunk, if this neuron has a spike
#which is defined by any value exceeds 80% of whole session values

fileName = 'OFT_ONOFF.csv'

randomShuffle_N = 5000 # random choose same ~ of frames from non focus part this times
thresholdP = 5 # to cut on/off among distribution

fps=20
Whole_list = ['W01']

## Call

In [3]:
for i in range(len(Whole_list)):

    mouse_name = Whole_list[i]
    print("processing ", mouse_name)
    
    # read in df
    #----------------------------------------------------------------------------------
    df_frame_label = pd.read_csv(os.path.join(root_path, mouse_name, 'trace_label.csv')).set_index('Frame', drop=True)
    

    # get omposite chunks
    #--------------------------------------------------------------------
    ce_compositeIntervals = find_compositeIntervals(df_frame_label,'center',
                                                    intervalLength,gapMax,if_CombineAdjacentChunks)
    ce_df = get_chunksInDF(ce_compositeIntervals, df_frame_label)
    co_compositeIntervals = find_compositeIntervals(df_frame_label,'corner',
                                                    intervalLength,gapMax,if_CombineAdjacentChunks)
    co_df = get_chunksInDF(co_compositeIntervals, df_frame_label)


    
    # get on/off list
    #--------------------------------------------------------------------
    centerON,centerOFF,allNeurons = classify_ONOFF(df_frame_label,ce_df,'center',fps,
                                          randomShuffle_N, Fpercentage,SBpercentage,
                                          thresholdP,os.path.join(root_path, mouse_name), 
                                          checkChance, ce_compositeIntervals, plotD=False)
    cornerON,cornerOFF,_ = classify_ONOFF(df_frame_label,co_df,'corner',fps,
                                          randomShuffle_N, Fpercentage,SBpercentage,
                                          thresholdP,os.path.join(root_path, mouse_name), 
                                          checkChance, co_compositeIntervals, plotD=False)
    print('    '+str(len(ce_compositeIntervals))+' centerEnters '+str(len(co_compositeIntervals))+' cornerEnters', end='')
    print(', found '+str(len(centerON))+' centerON | '+str(len(cornerON))+' cornerON')
    
    # generate summary
    all_ONOFF_neuron_list = [centerON, centerOFF, cornerON, cornerOFF]
    columns_name = ['CenterON','CenterOFF','CornerON','CornerOFF']
    
    # ON/OFF are saved to each mouse folder
    mouse_ONOFF = generate_ONOFF_Summary(allNeurons, all_ONOFF_neuron_list, columns_name, mouse_name, root_path, fileName)
    
    
    break

mouse_ONOFF

processing  W01
    2 centerEnters 29 cornerEnters, found 11 centerON | 17 cornerON


,CenterON,CenterOFF,CornerON,CornerOFF
C000,0,0,0,0
C001,0,0,0,1
C002,0,0,0,0
C004,0,0,0,0
C005,0,0,0,0
...,...,...,...,...
C096,0,0,0,0
C097,0,0,1,0
C099,0,0,0,1
C100,0,0,0,0
